# Beijing PM2.5 Forecasting  
## Notebook 01 — Data Loading & Initial Preparation

In this notebook, we load the **Beijing Aotizhongxin air quality dataset**, which contains hourly air pollution and meteorological measurements.

Our goals here are simple and deliberate:
- Load the raw dataset correctly
- Create a proper **datetime index**
- Keep **all pollutants and weather variables**
- Save a clean baseline dataset for later notebooks

No modeling, no imputation, no feature engineering yet. This notebook is only about **getting the data right**.


In [1]:
import pandas as pd
import numpy as np

# Make DataFrame outputs easier to read in the notebook
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## 1. Load the Raw Aotizhongxin Dataset

The Beijing dataset contains **12 monitoring stations**. For this project, we start with **Aotizhongxin**, one of the most commonly used stations in academic research.

We will work with **one station first** to keep the analysis clean and interpretable.

In [13]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data/raw/Dataset/PRSA2017_Data/PRSA_Data_")

# Find the Aotizhongxin station file robustly
file_path = next(DATA_DIR.glob("*Aotizhongxin*.csv"))

print("Loading file from:", file_path)

df = pd.read_csv(file_path)
df.head()

Loading file from: ..\data\raw\Dataset\PRSA2017_Data\PRSA_Data_\PRSA_Data_Aotizhongxin.csv


,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
0,1,2013,3,1,0,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4,Aotizhongxin
1,2,2013,3,1,1,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,N,4.7,Aotizhongxin
2,3,2013,3,1,2,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,NNW,5.6,Aotizhongxin
3,4,2013,3,1,3,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,NW,3.1,Aotizhongxin
4,5,2013,3,1,4,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,N,2.0,Aotizhongxin


## 2. Inspect Dataset Structure

Before touching anything, we check:
- Number of rows and columns
- Column names
- Whether all expected variables are present

This step helps catch data issues early.

In [14]:
print("Dataset shape:", df.shape)
df.columns

Dataset shape: (35064, 18)


Index(['No', 'year', 'month', 'day', 'hour', 'PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN',
       'wd', 'WSPM', 'station'],
      dtype='object')

## 3. Remove Non-Informative Columns

Some columns are not useful for modeling:
- `No` → just a row counter
- `station` → constant value for this file

Removing them early keeps the dataset tidy.

In [15]:
# Drop columns that do not add predictive value
df = df.drop(columns=["No", "station"])

## 4. Create a Datetime Index (Very Important)

The dataset stores time as separate columns:
`year`, `month`, `day`, `hour`.

For time-series modeling, we **must combine these into a single datetime index**.
This is a critical step for:
- Time-based splits
- Lag features
- Forecasting models (LSTM, XGBoost)

In [16]:
# Combine year, month, day, and hour into a single datetime column
df["datetime"] = pd.to_datetime(df[["year", "month", "day", "hour"]])

# Remove the original time columns
df = df.drop(columns=["year", "month", "day", "hour"])

# Set datetime as index and ensure chronological order
df = df.set_index("datetime")
df = df.sort_index()

df.head()

,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM
datetime,,,,,,,,,,,,
2013-03-01 00:00:00,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4
2013-03-01 01:00:00,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,N,4.7
2013-03-01 02:00:00,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,NNW,5.6
2013-03-01 03:00:00,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,NW,3.1
2013-03-01 04:00:00,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,N,2.0


## 5. Checks on Time Index

We verify:
- Start and end timestamps
- That the data is continuous and ordered

This confirms we truly have **hourly time-series data**.

In [17]:
print("Start date:", df.index.min())
print("End date:", df.index.max())
print("Total hourly records:", len(df))

Start date: 2013-03-01 00:00:00
End date: 2017-02-28 23:00:00
Total hourly records: 35064


## 6. Missing Values Overview

Missing values are **expected** in real air quality data. We intentionally **do not fix them here**.

These missing values will be removed in another notebook (**02.5_cleaning.ipynb**).

In [18]:
# Count missing values per column
df.isna().sum().sort_values(ascending=False)

CO       1776
O3       1719
NO2      1023
SO2       935
PM2.5     925
PM10      718
wd         81
TEMP       20
PRES       20
DEWP       20
RAIN       20
WSPM       14
dtype: int64

In [19]:
# Calculate percentage of missing values per column
na_count = df.isna().sum()
na_percentage = (na_count / len(df)) * 100

na_summary = (
    pd.DataFrame({
        "Missing Count": na_count,
        "Missing Percentage (%)": na_percentage.round(2)
    })
    .sort_values("Missing Percentage (%)", ascending=False)
)

na_summary

,Missing Count,Missing Percentage (%)
CO,1776,5.07
O3,1719,4.90
NO2,1023,2.92
SO2,935,2.67
PM2.5,925,2.64
PM10,718,2.05
wd,81,0.23
TEMP,20,0.06
PRES,20,0.06
DEWP,20,0.06


## 7. Feature Groups Used in This Project

We will:
- **Predict**: PM2.5
- **Use as features**:
  - Other pollutants (PM10, SO₂, NO₂, CO, O₃)
  - Weather variables (temperature, pressure, wind, etc.)

This reflects real-world air quality forecasting systems.

In [21]:
pollutant_features = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"]
weather_features = ["TEMP", "PRES", "DEWP", "RAIN", "WSPM", "wd"]

print("Pollutant features:", pollutant_features)
print("Weather features:", weather_features)

Pollutant features: ['PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3']
Weather features: ['TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM', 'wd']


## 8. Save Clean Baseline Dataset

We now save this dataset exactly as it is:
- Correct datetime index
- All features included
- Missing values untouched

This file becomes the **starting point** for all later notebooks.

In [22]:
OUTPUT_PATH = "../data/processed/aotizhongxin_baseline.csv"

df.to_csv(OUTPUT_PATH)

print(f"Baseline dataset saved to: {OUTPUT_PATH}")

Baseline dataset saved to: ../data/processed/aotizhongxin_baseline.csv


### Summary

By the end of this notebook, we have:
- Loaded the correct Beijing station data
- Built a proper hourly datetime index
- Included all pollutants and weather variables
- Preserved real-world missing values
- Created a reusable baseline dataset

From here onward, we can focus purely on **analysis and modeling**.